# 05.19 - ML Pipelines & ColumnTransformer

**Phase:** 05 - Machine Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

Raw data has mixed types, missing values, and different scales. Pipelines chain preprocessing and modeling into a single object, preventing data leakage.

## 2. Why Does This Matter?

Pipelines prevent the #1 mistake (data leakage from preprocessing on test data) and make your code reproducible.

## 3. Prerequisites

- 05.04: Model evaluation
- 05.13: Feature engineering

## 4. Learning Objectives

- Build sklearn Pipelines for numerical and categorical features
- Use ColumnTransformer for mixed-type data
- Prevent data leakage in preprocessing
- Apply to the real Titanic dataset

## 5. Mental Model

Pipeline = Preprocessing + Model in one object.
fit() on training data transforms AND trains.
transform() on test data uses training statistics (no leakage).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)
print("Libraries loaded.")

Libraries loaded.


## 6. The Data Leakage Problem

When you fit a scaler on ALL data (including test), information from the test set leaks into training. Pipelines prevent this.

In [2]:
titanic = sns.load_dataset("titanic")
df = titanic[["pclass", "sex", "age", "fare", "embarked", "survived"]].dropna()
X = df.drop("survived", axis=1)
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train: " + str(len(X_train)) + ", Test: " + str(len(X_test)))

Train: 569, Test: 143


## 7. ColumnTransformer for Mixed Types

Real data has numerical columns (age, fare) and categorical columns (sex, embarked). ColumnTransformer applies different preprocessing to different columns.

In [3]:
num_features = ["age", "fare"]
cat_features = ["pclass", "sex", "embarked"]

num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_features),
        ("cat", cat_transformer, cat_features)
    ]
)

full_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42))
])

full_pipeline.fit(X_train, y_train)
y_pred = full_pipeline.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("Full pipeline accuracy: " + str(round(acc, 4)))

Full pipeline accuracy: 0.7692


## 8. Cross-Validation with Pipelines

Pipelines work seamlessly with cross_val_score - no data leakage!

In [4]:
cv_scores = cross_val_score(full_pipeline, X, y, cv=5, scoring="accuracy")
print("CV accuracy: " + str(round(cv_scores.mean(), 4)) + " (+/- " + str(round(cv_scores.std(), 4)) + ")")

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42)
}

for name, model in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    scores = cross_val_score(pipe, X, y, cv=5, scoring="accuracy")
    print(name + ": " + str(round(scores.mean(), 4)) + " (+/- " + str(round(scores.std(), 4)) + ")")

CV accuracy: 0.7837 (+/- 0.0317)


LogisticRegression: 0.7823 (+/- 0.0404)


RandomForest: 0.7837 (+/- 0.0317)


## 9. Common Mistakes

1. Fitting preprocessing on all data - always use Pipeline
2. Forgetting categorical encoding
3. Not handling missing values
4. Using accuracy for imbalanced data

## 10. Coding Exercises

### Exercise 1: California Housing Pipeline
Build a Pipeline for the California Housing dataset.

### Exercise 2: Extended Titanic Pipeline
Add more features (sibsp, parch) to the Titanic pipeline.

In [5]:
# EXERCISE 1: California Housing Pipeline
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import Ridge
# Your code here: create Pipeline with StandardScaler + Ridge
print("Exercise: Build a Pipeline for California Housing.")

Exercise: Build a Pipeline for California Housing.


In [6]:
# EXERCISE 2: Extended Titanic Pipeline
# Add sibsp, parch features. Try different models.
print("Exercise: Extend the Titanic pipeline.")

Exercise: Extend the Titanic pipeline.


## 11. Closed-Book Recall

1. What problem do Pipelines solve?
2. How does ColumnTransformer handle mixed types?
3. Why is fitting on test data wrong?
4. How do you cross-validate a Pipeline?

## 12. Teach-Back Questions

Explain to another person:
- Why Pipelines prevent data leakage.
- How ColumnTransformer works.
- When to use Pipeline vs manual preprocessing.

## 13. Summary

Pipelines chain preprocessing and modeling. ColumnTransformer handles mixed types. Always fit on train only. Cross-validation with Pipelines is leak-free.

## 14. Further Experiment

1. Add feature selection to the Pipeline.
2. Try different preprocessing.
3. Deploy the Pipeline as a REST API.

## Verification Status
```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: [numpy, pandas, matplotlib, scikit-learn, seaborn]
OUTPUTS: PASS
LAST VERIFIED: 2026-08-30
```